In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report
import numpy as np

# Load the dataset
df = pd.read_csv('../MachineLearningRating_v3.txt', delimiter='|', encoding='utf-8', low_memory=False)
df.head()

,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,ExcessSelected,CoverCategory,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims
0,145249,12827,2015-03-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
1,145249,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
2,145249,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0
3,145255,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0
4,145255,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0


In [8]:
# Filter for regression task
df_claims = df[df['TotalClaims'] > 0].copy()
df['HasClaim'] = df['TotalClaims'] > 0

In [9]:
# Separate features and targets
X_reg = df_claims.drop(columns=['TotalClaims', 'TotalPremium'])
y_reg = df_claims['TotalClaims']

X_cls = df.drop(columns=['TotalClaims', 'TotalPremium'])
y_cls = df['HasClaim'].astype(int)

# Identify numerical and categorical columns
num_cols = X_reg.select_dtypes(include='number').columns.tolist()
cat_cols = X_reg.select_dtypes(include='object').columns.tolist()


In [10]:

# Preprocessor pipeline
preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='mean'), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])

In [11]:
#Model Training & Evaluation

### 1. Claim Severity Model (Regression)
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)


In [12]:
models_reg = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(n_estimators=100),
    'XGBoost': XGBRegressor(objective='reg:squarederror')
}


In [15]:
from sklearn.metrics import mean_squared_error, r2_score
for name, model in models_reg.items():
    pipeline = Pipeline([('pre', preprocessor), ('model', model)])
    pipeline.fit(X_train_r, y_train_r)
    preds = pipeline.predict(X_test_r)
    print(f"--- {name} ---")
    print("RMSE:", mean_squared_error(y_test_r, preds))
    print("R2:", r2_score(y_test_r, preds))

c:\Users\PC\car_insurance_risk_analytics\.venv\Lib\site-packages\sklearn\impute\_base.py:637: UserWarning: Skipping features without any observed values: ['NumberOfVehiclesInFleet']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
c:\Users\PC\car_insurance_risk_analytics\.venv\Lib\site-packages\sklearn\impute\_base.py:637: UserWarning: Skipping features without any observed values: ['CrossBorder']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
c:\Users\PC\car_insurance_risk_analytics\.venv\Lib\site-packages\sklearn\impute\_base.py:637: UserWarning: Skipping features without any observed values: ['NumberOfVehiclesInFleet']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
c:\Users\PC\car_insurance_risk_analytics\.venv\Lib\site-packages\sklearn\impute\_base.py:637: UserWarning: Skipping features without any observed values: ['CrossBorder']

--- LinearRegression ---
RMSE: 1365141071.1017983
R2: 0.1511625817255816


c:\Users\PC\car_insurance_risk_analytics\.venv\Lib\site-packages\sklearn\impute\_base.py:637: UserWarning: Skipping features without any observed values: ['NumberOfVehiclesInFleet']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
c:\Users\PC\car_insurance_risk_analytics\.venv\Lib\site-packages\sklearn\impute\_base.py:637: UserWarning: Skipping features without any observed values: ['CrossBorder']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
c:\Users\PC\car_insurance_risk_analytics\.venv\Lib\site-packages\sklearn\impute\_base.py:637: UserWarning: Skipping features without any observed values: ['NumberOfVehiclesInFleet']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
c:\Users\PC\car_insurance_risk_analytics\.venv\Lib\site-packages\sklearn\impute\_base.py:637: UserWarning: Skipping features without any observed values: ['CrossBorder']

--- RandomForest ---
RMSE: 1360162169.5109346
R2: 0.1542584361113999
--- XGBoost ---
RMSE: 1647555280.5112793
R2: -0.024441063548730435


c:\Users\PC\car_insurance_risk_analytics\.venv\Lib\site-packages\sklearn\impute\_base.py:637: UserWarning: Skipping features without any observed values: ['NumberOfVehiclesInFleet']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
c:\Users\PC\car_insurance_risk_analytics\.venv\Lib\site-packages\sklearn\impute\_base.py:637: UserWarning: Skipping features without any observed values: ['CrossBorder']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(


In [16]:
#Claim Probability Model (Classification)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)


In [ ]:
models_cls = {
    'RandomForest': RandomForestClassifier(n_estimators=100),
    'XGBoost': XGBClassifier()
}

In [ ]:
print(df.isnull().sum())  # Identify columns with all NaNs

In [ ]:
imputer = SimpleImputer(strategy='constant', fill_value='Unknown')

In [ ]:
imputer = SimpleImputer(strategy='median')

In [ ]:
df = df.drop(columns=["NumberOfVehiclesInFleet"])

In [ ]:
for name, model in models_cls.items():
    pipeline = Pipeline([('pre', preprocessor), ('model', model)])
    pipeline.fit(X_train_c, y_train_c)
    preds = pipeline.predict(X_test_c)
    print(f"--- {name} ---")
    print("Accuracy:", accuracy_score(y_test_c, preds))
    print(classification_report(y_test_c, preds))

In [ ]:
#Feature Importance with SHAP
import shap
model = XGBRegressor(objective='reg:squarederror')
pipeline = Pipeline([('pre', preprocessor), ('model', model)])
pipeline.fit(X_train_r, y_train_r)

In [ ]:
# SHAP values (on raw data)
X_preprocessed = preprocessor.fit_transform(X_train_r)
explainer = shap.Explainer(model)
shap_values = explainer(X_preprocessed)

In [ ]:
shap.summary_plot(shap_values, X_preprocessed, show=False)